# insurance-optimise

**Constrained portfolio rate optimisation for UK personal lines insurance.**

This notebook shows the core workflow in under two minutes: build a synthetic renewal book, set up FCA-compliant constraints, run the optimiser, and trace the profit-retention efficient frontier.

The problem this solves: your technical model tells you the right risk price, but the commercial question is different - which multiplier maximises expected profit for each policy, subject to the ENBP ceiling (FCA PS21/11), a loss ratio cap, a retention floor, and maximum rate-change limits? Solving this as a constrained optimisation problem consistently outperforms the spreadsheet scenario approach.

In [ ]:
!pip install -q insurance-optimise polars numpy scipy

## Synthetic renewal book

500 policies. 70% are renewals (ENBP constraint applies). Technical price ranges from £300 to £1,200, representing the range of a typical UK motor portfolio. Price elasticity varies by policy - younger drivers are more price-sensitive.

In [ ]:
import numpy as np
import polars as pl

rng = np.random.default_rng(42)
n   = 500

technical_price    = rng.uniform(300, 1_200, n)
expected_loss_cost = technical_price * rng.uniform(0.55, 0.75, n)
p_renewal          = rng.uniform(0.70, 0.95, n)           # renewal probability at current price
price_elasticity   = rng.uniform(-2.5, -0.8, n)           # varies across book
is_renewal         = rng.choice([True, False], n, p=[0.7, 0.3])

# ENBP: FCA PS21/11 ceiling - the equivalent new business price for renewals
# Must be >= technical_price so it is a meaningful constraint
enbp = technical_price * rng.uniform(1.05, 1.30, n)

print(f"Policies:        {n}")
print(f"Renewals:        {is_renewal.sum()} ({is_renewal.mean():.0%})")
print(f"Avg tech price:  £{technical_price.mean():,.0f}")
print(f"Avg loss ratio:  {(expected_loss_cost / technical_price).mean():.1%}")
print(f"Avg elasticity:  {price_elasticity.mean():.2f}")

## Set up constraints and optimise

`ConstraintConfig` holds every regulatory and business constraint. The key ones here:

- `lr_max=0.70` - hard loss ratio ceiling
- `retention_min=0.85` - renewal policies must achieve at least 85% renewal rate
- `max_rate_change=0.20` - no individual policy moves more than 20% from its current price
- `enbp_buffer=0.01` - 1% safety margin below the ENBP ceiling (FCA PS21/11)
- `technical_floor=True` - price must be at or above technical price (no underwriting cross-subsidy)

In [ ]:
from insurance_optimise import PortfolioOptimiser, ConstraintConfig

config = ConstraintConfig(
    lr_max=0.70,
    retention_min=0.85,
    max_rate_change=0.20,
    enbp_buffer=0.01,
    technical_floor=True,
)

opt = PortfolioOptimiser(
    technical_price=technical_price,
    expected_loss_cost=expected_loss_cost,
    p_demand=p_renewal,
    elasticity=price_elasticity,
    renewal_flag=is_renewal,
    enbp=enbp,
    constraints=config,
)

result = opt.optimise()
print(result)

## Interpreting the result

The `OptimisationResult` gives you:
- `multipliers` - the optimal price multiplier for each policy (applied to technical_price)
- `new_premiums` - the optimal premium for each policy
- `expected_profit` and `expected_loss_ratio` - portfolio-level outcomes
- `expected_retention` - renewal book retention at the optimal prices
- `shadow_prices` - marginal cost of each binding constraint; a large value means that constraint is the main thing limiting profit

In [ ]:
print(f"Converged:          {result.converged}")
print(f"Expected profit:    £{result.expected_profit:,.0f}")
print(f"Expected GWP:       £{result.expected_gwp:,.0f}")
print(f"Expected LR:        {result.expected_loss_ratio:.3f}")
if result.expected_retention is not None:
    print(f"Expected retention: {result.expected_retention:.1%}")
print(f"Solver iterations:  {result.n_iter}")
print()
print("Shadow prices (binding constraints):")
for name, val in result.shadow_prices.items():
    print(f"  {name}: {val:.4f}")
print()
print("Per-policy summary (first 5):")
print(result.summary_df.head())

## Efficient frontier

The frontier answers the question pricing teams actually need answered: "if we accept a lower retention target, how much more profit can we extract?" Run this before the pricing review to frame the conversation quantitatively.

The `EfficientFrontierResult.data` attribute is a Polars DataFrame with one row per sweep point.

In [ ]:
from insurance_optimise import EfficientFrontier

frontier = EfficientFrontier(
    optimiser=opt,
    sweep_param="volume_retention",
    sweep_range=(0.80, 0.95),
    n_points=8,
)
frontier_result = frontier.run()

# .data is a Polars DataFrame: one row per retention target
print(frontier_result.data.select(["epsilon", "profit", "gwp", "loss_ratio", "retention", "converged"]))

## FCA audit trail

Under Consumer Duty, you need to demonstrate that your pricing delivers fair value. Under PS21/11, ENBP enforcement must be evidenced. The audit trail captures every input, constraint, and solution so you can show the FCA exactly what the optimiser was configured to do and what it produced.

In [ ]:
import json, tempfile, os

with tempfile.NamedTemporaryFile(suffix=".json", delete=False, mode="w") as f:
    audit_path = f.name

result.save_audit(audit_path)

with open(audit_path) as f:
    audit = json.load(f)

print("Audit trail top-level keys:", list(audit.keys()))
print()
print("Constraint config stored in audit:")
print(json.dumps(audit["constraint_config"], indent=2))

os.unlink(audit_path)

## Next steps

- **Scenario analysis**: use `opt.optimise_scenarios([elast_pessimistic, elast_central, elast_optimistic])` to show the spread across elasticity uncertainty
- **Stochastic LR**: set `stochastic_lr=True` with `ClaimsVarianceModel` to use the Branda (2014) chance-constrained formulation
- **Demand curves**: the `insurance-demand` library fits conversion and retention models whose callables feed directly into this library

**GitHub:** https://github.com/burning-cost/insurance-optimise  
**PyPI:** https://pypi.org/project/insurance-optimise/